# Social media — donation value (regression) & referral presence (classification)


## 1. Business Understanding

**BOTH.** (1) **EXPLANATORY OLS** — interpret which post attributes **associate with** higher `estimated_donation_value_php`. (2) **PREDICTIVE Gradient Boosting** — forecast value for **post planning**. OLS prioritizes *inference* under linearity assumptions; GBR prioritizes *out-of-sample* fit — we contrast explicitly.

**Success:** GBR test MAE minimized; OLS significant coefficients reviewed for direction.


## 2. Data Understanding & Preparation (EDA)


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
import warnings; warnings.filterwarnings('ignore')

from pathlib import Path
import subprocess, sys, warnings
warnings.filterwarnings("ignore")

def _find_ml_dir() -> Path:
    p = Path.cwd().resolve()
    for _ in range(10):
        b = p / "build_master_datasets.py"
        d = p / "data" / "supporters.csv"
        if b.exists() and d.exists():
            return p
        v2 = p / "ml-pipelines-v2"
        if (v2 / "build_master_datasets.py").exists():
            return v2
        p = p.parent
    raise FileNotFoundError("Could not find ml-pipelines-v2 — open from repo or set cwd to ml-pipelines-v2/")

ML_DIR = _find_ml_dir()
DATA_DIR = ML_DIR / "data"
MODEL_DIR = ML_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)
BUILDER = ML_DIR / "build_master_datasets.py"
if BUILDER.exists() and (
    not (DATA_DIR / "donor_master.csv").exists()
    or not (DATA_DIR / "resident_master.csv").exists()
):
    subprocess.run([sys.executable, str(BUILDER)], check=True)

posts = pd.read_csv(DATA_DIR / 'social_media_posts.csv', low_memory=False)
posts['target_val'] = posts['estimated_donation_value_php'].fillna(0)
posts['target_ref'] = (posts['donation_referrals'].fillna(0) > 0).astype(int)
feat = ['platform','post_type','media_type','sentiment_tone','content_topic','post_hour','day_of_week',
        'is_boosted','num_hashtags','has_call_to_action','features_resident_story','caption_length','engagement_rate']
posts['is_boosted'] = posts['is_boosted'].astype(str).str.lower().eq('true').astype(int)
posts['has_call_to_action'] = posts['has_call_to_action'].astype(str).str.lower().eq('true').astype(int)
posts['features_resident_story'] = posts['features_resident_story'].astype(str).str.lower().eq('true').astype(int)
print(posts.shape)
print(posts[feat + ['target_val','target_ref']].describe())


In [ ]:
sns.heatmap(posts[['post_hour','num_hashtags','caption_length','engagement_rate','target_val']].corr(), annot=True)
plt.show()
for c in ['platform','post_type','content_topic']:
    ct = pd.crosstab(posts[c], posts['target_ref'])
    print(c, stats.chi2_contingency(ct)[:2])


In [ ]:
m = posts.dropna(subset=['target_val'])
num_f = ['post_hour','num_hashtags','caption_length','engagement_rate','is_boosted','has_call_to_action','features_resident_story']
cat_f = ['platform','post_type','media_type','sentiment_tone','content_topic','day_of_week']
for c in cat_f:
    m[c] = m[c].fillna('Unknown').astype(str)
for c in num_f:
    m[c] = pd.to_numeric(m[c], errors='coerce')
    m[c] = m[c].fillna(m[c].median())


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
X = m[num_f + cat_f]
y = m['target_val']
y_cls = m['target_ref']
X_train, X_test, y_train, y_test, y_train_c, y_test_c = train_test_split(
    X, y, y_cls, test_size=0.2, random_state=42)
numeric_t = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
categorical_t = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
preprocess = ColumnTransformer([('num', numeric_t, num_f), ('cat', categorical_t, cat_f)])


## 3. Modeling & Feature Selection


In [ ]:
import statsmodels.api as sm
X_ols = pd.get_dummies(X_train[num_f + cat_f], drop_first=True)
X_ols = X_ols.apply(pd.to_numeric, errors='coerce').fillna(0.0)
X_ols = sm.add_constant(X_ols, has_constant='add')
ols = sm.OLS(np.asarray(y_train, dtype=float), np.asarray(X_ols, dtype=float)).fit(cov_type='HC3')
print(ols.summary().tables[1])


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
gbr = Pipeline([('prep', preprocess), ('reg', GradientBoostingRegressor(random_state=42))])
param = {'reg__n_estimators':[100,200],'reg__max_depth':[2,3,4],'reg__learning_rate':[0.05,0.1]}
rs = RandomizedSearchCV(gbr, param, n_iter=8, cv=3, random_state=42, n_jobs=1, scoring='neg_mean_absolute_error')
rs.fit(X_train, y_train)
best = rs.best_estimator_
pred = best.predict(X_test)
print('MAE', mean_absolute_error(y_test, pred), 'RMSE', mean_squared_error(y_test, pred)**0.5, 'R2', r2_score(y_test, pred))


In [ ]:
fi = best.named_steps['reg'].feature_importances_
names = best.named_steps['prep'].get_feature_names_out()
pd.Series(fi, index=names).sort_values(ascending=False).head(15)


## 4. Evaluation & Interpretation

Compare OLS (inference) vs GBR (prediction). **Business:** Comms team uses GBR for **expected PHP**; leadership reads OLS for **directional** levers — noting **association ≠ causation**.


In [ ]:
from sklearn.linear_model import LogisticRegression
pipe_cls = Pipeline([('prep', preprocess), ('clf', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42))])
pipe_cls.fit(X_train, y_train_c)
from sklearn.metrics import roc_auc_score
print('Referral>0 ROC-AUC', roc_auc_score(y_test_c, pipe_cls.predict_proba(X_test)[:,1]))
print('GBR test MAE', mean_absolute_error(y_test, pred))
print('OLS train R2', ols.rsquared)


## 5. Causal / Relationship Analysis

The **OLS + HC3** specification is our **explicit associational** model; we avoid claiming that changing a post feature *causes* donations without experimental design.


## 6. Deployment Notes

**social_model.sav** powers a **Post Optimizer** widget on the **Social Media** admin page.


In [ ]:
import joblib
joblib.dump(best, MODEL_DIR / 'social_model.sav')
model = joblib.load(MODEL_DIR / 'social_model.sav')
print(model.predict(X_test.iloc[:1]))
